<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-08-agents-and-adk/lesson-8.3-agent-engine/notebooks/GCP_Capstone_8.3_AgentEngine.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.3 Agent Engine — A Managed Runtime for an Agent Outside the Kit: Memory, Context, Identity, Cost
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

> **Naming, 2026:** Vertex AI Agent Engine is now **Agent Runtime** (Gemini Enterprise Agent Platform). The Python surface did not move: `vertexai.Client(...).agent_engines`. Read "Agent Engine" below as Agent Runtime.

An agent that will run on a managed runtime, not on Cloud Run and not in this notebook, so it reaches the lane the only way an agent outside the kit should: through `documind-mcp`, with a credential minted per call and the roster deciding. Memory wiring, context caching and compaction come from ADK; the runtime brings Sessions and Memory Bank. The agent runs locally first, asserted, and deploys under its own named account only when the owner flips the switch, because a runtime bills until it is deleted.


## Setup
No kit import, on purpose. Two locations: the runtime is regional, the model is global.


In [ ]:
!pip install -q "google-cloud-aiplatform[agent-engines,adk]==2.1.0" "google-adk[mcp]==2.8.0" google-genai==2.22.0 google-auth==2.57.1

# aiplatform 2.x, not the 1.153 this lesson once pinned: the Agent Engine client is
# vertexai.Client(...).agent_engines, the same surface, and 2.1.0 is what 4.7 already installs.
# [adk] constrains google-adk itself (>=1.27,<3), so the two pins are checked against each other.

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"           # where the Agent Engine RUNTIME lives (regional); the model runs global

import os
import google.auth
from google.auth.transport.requests import AuthorizedSession

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"      # Gemini 3.x generation is served from the global endpoint
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

creds, _ = google.auth.default()
NUMBER   = AuthorizedSession(creds).get(f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
MCP_URL  = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"          # 7.2's server: the ONLY thing this agent knows about DocuMind
AGENT_SA = f"documind-agent-sa@{PROJECT_ID}.iam.gserviceaccount.com"  # the account the deployed agent runs as (sa.tf, 8.4): on acme's roster
MEMBER_SA = f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com"    # what THIS notebook mints as, until the agent has its own account

# NOTHING FROM THE KIT IS IMPORTED HERE, on purpose. This agent will run on a managed runtime that is
# not Cloud Run and not this notebook: it reaches the lane the way any agent outside the kit does,
# through the MCP server, with the identity rules 7.1 built. tools/check_contract.py fails this
# notebook if a `from shared import` ever appears.
print("runtime region:", REGION, "| MCP:", MCP_URL)


## Cell 1: The credential, in both homes
One function: on the runtime it mints from the agent's own service account; in this notebook it impersonates a roster member. Called per tool call, never captured.


In [ ]:
SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]


def mcp_headers(readonly_context=None) -> dict:
    """The credential for documind-mcp, minted on EVERY call - ADK calls a header_provider per tool call.

    Two homes, one function. On the managed runtime the agent has a service account and google-auth
    can mint an ID token for the MCP url from it (fetch_id_token). In this notebook there is no such
    account, so the same function impersonates a roster member instead (7.1's hook, spelled out).
    The audience is the server's ROOT url: Cloud Run checks it, and the server checks it again.
    """
    import google.auth
    import google.auth.transport.requests
    import google.oauth2.id_token
    from google.auth import impersonated_credentials

    request = google.auth.transport.requests.Request()
    impersonate = os.environ.get("DOCUMIND_IMPERSONATE_SA")
    if impersonate:
        source, _ = google.auth.default()
        target = impersonated_credentials.Credentials(source_credentials=source, target_principal=impersonate, target_scopes=SCOPE)
        idc = impersonated_credentials.IDTokenCredentials(target, target_audience=MCP_URL, include_email=True)
        idc.refresh(request)
        return {"Authorization": f"Bearer {idc.token}"}
    return {"Authorization": f"Bearer {google.oauth2.id_token.fetch_id_token(request, MCP_URL)}"}


os.environ["DOCUMIND_IMPERSONATE_SA"] = MEMBER_SA      # the notebook's leg; unset on the runtime
print("token minted, first 24 chars:", mcp_headers()["Authorization"][7:31], "...")


## Cell 2: The agent, with memory wiring
`PreloadMemoryTool` recalls at the top of every turn; the `after_agent_callback` saves the session to memory when the turn ends. The lane's tools come from the MCP server, prefixed.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.tools.preload_memory_tool import PreloadMemoryTool


async def save_memory(callback_context: CallbackContext):
    """after_agent_callback: hand the finished session to the memory service. On the runtime that is
    Memory Bank, which extracts durable facts on its own; locally it is a dict, which remembers verbatim."""
    await callback_context.add_session_to_memory()
    return None


lane = McpToolset(
    # timeout=120: ADK's default is 5 s; a retrieve() through the server is rag-api plus a Gemini
    # answer, up to 90 s cold. The first live peer passed the instant refusal and failed the real
    # question on the default - a timeout that only bites when the tool works is the worst kind.
    connection_params=StreamableHTTPConnectionParams(url=f"{MCP_URL}/mcp", timeout=120, sse_read_timeout=300),
    header_provider=mcp_headers,                 # per call, never captured (7.3)
    tool_name_prefix="docs",
)

INSTRUCTION = (
    "You are DocuMind, answering questions about the company's documents. For any question about the "
    "documents call docs_retrieve, answer only from what it returns and cite the sources it names; pass "
    "tenant='acme' to docs_retrieve, docs_list_documents and docs_corpus_stats. Use what you remember about "
    "this user when it is relevant, and say what you remembered. If a tool refuses or the corpus cannot "
    "answer, say so and cite nothing."
)

root_agent = LlmAgent(
    name="documind_managed",
    model="gemini-3.6-flash",
    instruction=INSTRUCTION,
    tools=[PreloadMemoryTool(), lane],          # recall at the top of every turn, the lane's tools for the rest
    after_agent_callback=save_memory,
)
print("agent:", root_agent.name, "| tools: PreloadMemoryTool + the MCP server's four, prefixed docs_")


## Cell 3: The App - caching and compaction


In [ ]:
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.agents.context_cache_config import ContextCacheConfig

app = App(
    name="documind",
    root_agent=root_agent,
    context_cache_config=ContextCacheConfig(
        min_tokens=4096,      # the model's own floor for Gemini 3 - 2048 configures a cache that never forms
        ttl_seconds=600,
        cache_intervals=5,
    ),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,   # fold every three events into a summary; keep one raw event of overlap
        overlap_size=1,
    ),
)
print("App configured: caching floor", app.context_cache_config.min_tokens, "tokens | compaction every", app.events_compaction_config.compaction_interval, "events")

# min_tokens is a floor on the PRIOR request's tokens before a cache is worth forming, and the model
# sets the real minimum: ADK's own docstring says 2048 for Gemini 2.5 and 4096 for Gemini 3. Nothing
# is cached on the first request of a session, because there is no previous count yet.


## Cell 4: Run it here first, asserted
Local services stand in for the runtime's; the MCP server is the deployed one. Turn 2 is a new session that remembers turn 1.


In [ ]:
from google.adk.memory import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# THE SAME AGENT, RUN HERE, BEFORE IT IS DEPLOYED. Local services stand in for the runtime's:
# InMemorySessionService for Sessions, InMemoryMemoryService for Memory Bank. The MCP server is
# the deployed one, so the retrieval and the identity check are real; only the memory is a dict.
session_service, memory_service = InMemorySessionService(), InMemoryMemoryService()
runner = Runner(app=app, session_service=session_service, memory_service=memory_service)
LAST = {"calls": [], "results": [], "text": ""}


async def run_turn(question: str, user_id: str = "priya", session_id: str | None = None) -> bool:
    LAST.update(calls=[], results=[], text="")
    if session_id is None:
        session_id = (await session_service.create_session(app_name="documind", user_id=user_id)).id
    content = types.Content(role="user", parts=[types.Part.from_text(text=question)])
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                LAST["calls"].append(part.function_call.name)
                print(f"  [tool call] {part.function_call.name}({dict(part.function_call.args or {})})")
            if part.function_response:
                LAST["results"].append(part.function_response.response)
                print(f"  [tool result] {str(part.function_response.response)[:150]}")
            if part.text:
                LAST["text"] += part.text
                print(f"  [agent] {part.text.strip()[:350]}")
    return bool([c for c in LAST["calls"] if c.startswith("docs_")])


# Turn 1, session A: the gratuity row (lk-16) - through the MCP server, cited. ASSERTED.
called = await run_turn("After how many years of continuous service does gratuity become payable?")
assert called, "no docs_ tool call - the model answered from memory (is documind-mcp deployed? may you mint as ui-sa?)"

# Turn 2, a NEW session for the same user: the after_agent_callback saved session A to memory, and
# PreloadMemoryTool recalls it at the top of this turn. Watch for the recalled line in the answer.
await run_turn("What did I ask you about last time, and what was the answer?")
print("\nrecalled across sessions:", "gratuity" in LAST["text"].lower())


## Cell 5: Deploy, under a named account - the owner's switch


In [ ]:
import vertexai
from vertexai.agent_engines import AdkApp
from google import genai
from google.adk.models.google_llm import Gemini

DEPLOY = False        # the owner's switch: an Agent Engine runtime BILLS until it is deleted (Cell 6)

# TWO locations, and only ONE of them is settable here. The client location is where the runtime
# lives (regional, us-central1). The model's endpoint must be global for Gemini 3.x - and the deploy
# docs list GOOGLE_CLOUD_LOCATION among the variables you must NOT set in env_vars, so the model
# object carries its endpoint with it instead: ADK's own Gemini docstring says subclass and
# override api_client.


class GlobalGemini(Gemini):
    """A Gemini model pinned to the global endpoint, wherever the agent is deployed."""

    @property
    def api_client(self) -> genai.Client:
        return genai.Client(enterprise=True, location="global")


root_agent.model = GlobalGemini(model="gemini-3.6-flash")
client = vertexai.Client(project=PROJECT_ID, location=REGION)
adk_app = AdkApp(agent=root_agent, enable_tracing=True)

# WHO THE DEPLOYED AGENT IS. service_account names documind-agent-sa (terraform/sa.tf): invoker on
# documind-mcp, Gemini through aiplatform.user, on acme's roster and no other - so docs_retrieve
# needs no tenant argument and a request naming zeta is the server's refusal. Without this line the
# runtime uses its service agent, which is on no roster, and every retrieve is a 403 that reads as
# "the corpus cannot answer". One more grant, once, in Cloud Shell - the runtime's service agent
# must be allowed to act as the account:
#   gcloud iam service-accounts add-iam-policy-binding $AGENT_SA --role=roles/iam.serviceAccountUser \
#     --member=serviceAccount:service-$NUMBER@gcp-sa-aiplatform-re.iam.gserviceaccount.com
#
# STILL UNVERIFIED on 8 September: that the runtime's credential can mint an ID token for a Cloud Run
# audience (mcp_headers' fetch_id_token leg). The first deploy answers it; the fallback is to front
# documind-mcp with IAP and mint for its client id instead. The demo runs this agent locally (Cell 4).
if DEPLOY:
    remote = client.agent_engines.create(
        agent=adk_app,
        config={
            "display_name": "DocuMind managed agent",
            "requirements": ["google-cloud-aiplatform[agent-engines,adk]==2.1.0", "google-adk[mcp]==2.8.0",
                             "google-genai==2.22.0", "google-auth==2.57.1"],
            "staging_bucket": f"gs://{PROJECT_ID}-agent-staging",      # create it first: gcloud storage buckets create
            "env_vars": {"MCP_URL": MCP_URL, "GOOGLE_GENAI_USE_VERTEXAI": "TRUE"},
            "service_account": AGENT_SA,
            "min_instances": 0,                                         # scale-to-zero is the cost lever (Cell 8)
        },
    )
    print("deployed:", remote.api_resource.name)
else:
    print("DEPLOY=False: the agent above is what would be deployed; flip the switch when the owner says so")


## Cell 6: Query the deployed agent


In [ ]:
# The deployed agent, queried the way any client would - with the runtime's own Sessions, and its
# Memory Bank behind add_session_to_memory. Same question, same server, a different runtime.
if DEPLOY:
    session = await remote.async_create_session(user_id="priya")
    async for event in remote.async_stream_query(user_id="priya", session_id=session["id"],
                                                 message="After how many years of continuous service does gratuity become payable?"):
        print(str(event)[:300])
    await remote.async_add_session_to_memory(user_id="priya", session_id=session["id"])
    print("session saved to Memory Bank")
else:
    print("deploy first (Cell 5), then run this cell")


## Tear it down
A deployed runtime bills whether or not it is queried.


In [ ]:
# TEARDOWN - run this when you are done. A runtime bills for its compute whether or not anyone
# queries it; the most common way to spend money on this course is to deploy in a lesson and never
# come back. force=True also removes the child sessions and memories.
if DEPLOY:
    remote.delete(force=True)
    for e in client.agent_engines.list():
        print("still deployed:", e.api_resource.name)
    print("torn down")
else:
    print("nothing to tear down")


## Cell 7: HIPAA and DPDP, with the memory store included


In [ ]:
hipaa_checklist = {
    "1_baa": "Execute a BAA via Cloud Console > Privacy & Security > Legal",
    "2_encryption": "AES-256 at rest (default) + CMEK + TLS 1.2+ in transit",
    "3_network": "VPC Service Controls perimeter around Vertex AI resources",
    "4_access": "IAM least privilege + Cloud Audit Logs enabled - the agent's OWN account (Cell 5), never the service agent",
    "5_models": "Use only BAA-covered models (verify each model)",
    "6_region": "Pin regional services (Firestore, embeddings, Document AI, the runtime) to your compliant region. "
                "Gemini 3.x generation is served only from the global endpoint - verify your BAA covers it, or use a region-pinned model for PHI.",
    "7_dlp": "Sensitive Data Protection on data flows for PHI detection (5.5, 12.5)",
    "8_retention": "Configure zero data retention at project level; Memory Bank is a store of extracted facts - it needs a retention decision too",
}
print("HIPAA compliance checklist for DocuMind:")
for k, v in hipaa_checklist.items():
    print(f"  [{k}] {v}")

print("\nIndia DPDP Act:")
print("  - consent before processing personal data")
print("  - data principal rights: access, correction, erasure - and a memory store is personal data")
print("  - 72-hour breach notification")
print("  - use asia-south1 for Indian users' data at rest; the runtime and the model endpoint are separate decisions")
print("  - penalties: up to Rs 250 crore per violation")


## Cell 8: What the runtime costs


In [ ]:
# WHAT THE RUNTIME COSTS, honestly. Since 1 September 2026 Agent Runtime, Sessions and Memory Bank
# meter through Agent Compute (vCPU-hours) and Agent Storage (GiB-month); Memory Bank's own
# embedding and generation tokens bill at model rates on top. Take the rates from the live pricing
# page - these two are placeholders for the arithmetic, and the arithmetic is the lesson.
runtime_vcpu_hour = 0.085
runtime_gib_hour = 0.0090

flash_input_1m, flash_output_1m = 1.50, 7.50          # gemini-3.6-flash, standard rates (CLAUDE.md)
queries = 1000                                        # per month
in_tokens, out_tokens = 10_000, 1_000                 # per query, with the tool result in context

llm = queries * (in_tokens * flash_input_1m / 1e6 + out_tokens * flash_output_1m / 1e6)
always_on = 2 * runtime_vcpu_hour * 730               # min_instances=1, 2 vCPU: 730 hours a month
scale_to_zero = 2 * runtime_vcpu_hour * (queries * 8 / 3600)   # ~8 s of compute per query, nothing while idle

print(f"model, 1000 queries          : ${llm:8.2f}")
print(f"runtime, always on (min=1)   : ${always_on:8.2f}   <- {100 * always_on / (llm + always_on):.0f}% of the bill")
print(f"runtime, scale-to-zero       : ${scale_to_zero:8.2f}")
print(f"caching, input tokens at -90%: ${llm - 0.9 * queries * in_tokens * flash_input_1m / 1e6:8.2f}  (output and runtime unchanged)")
print("\nOn a low-traffic agent the dominant cost is a runtime billed by the hour whether or not anyone")
print("asks it anything. min_instances=0 is the lever; caching helps the model line only. And the")
print("Cloud Run peer in 8.4 answers the same question at Rs 0 while idle - which is the comparison to make.")


## Where this goes
- **8.4** deploys the same shape of agent - outside the kit, over the MCP server - on Cloud Run as an A2A peer, at Rs 0 while idle, and shows another agent calling it.
- **8.6** builds the memory you own (a store with a PII gate and an audit row) and says when to take Memory Bank instead.

## ✅ Lesson 8.3 complete
- ✅ An agent outside the kit: the MCP server is the only thing it knows about DocuMind
- ✅ A credential minted per call, in the notebook and on the runtime, from one function
- ✅ Memory wiring: recall at the top of a turn, save at the end; proven across two sessions locally
- ✅ Caching with the Gemini 3 floor, compaction with overlap
- ✅ A deploy under `documind-agent-sa`, gated by the owner; the still-unverified leg named
- ✅ The cost arithmetic: the runtime dominates a quiet agent, and scale-to-zero is the lever
